In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==========================================
# 1. 데이터 생성 파트 (공장)
# ==========================================
print("1. 레이더 신호 데이터를 생성하고 있습니다... (잠시만 기다려주세요)")

fs = 10e6       # 샘플링 주파수
T = 20e-6       # 펄스 길이
t = np.arange(0, T, 1/fs)
n_samples = len(t)

# 데이터셋 설정
n_per_class = 500  # 각 신호별로 500개씩 생성 (총 2000개)
classes = ['LFM', 'Barker', 'Costas', 'Frank']
X_data = [] # 이미지(스펙트로그램)을 담을 리스트
y_label = [] # 정답표(0, 1, 2, 3)를 담을 리스트

def get_signal(name):
    # (이전과 동일한 신호 생성 로직)
    if name == 'LFM':
        f_start, f_end = -2e6, 2e6
        return np.exp(1j * 2 * np.pi * (f_start * t + (f_end - f_start) / (2 * T) * t**2))
    elif name == 'Barker':
        code = np.array([1, 1, 1, 1, 1, -1, -1, 1, 1, -1, 1, -1, 1])
        chip = int(n_samples / len(code))
        phase = np.zeros(n_samples); 
        for i, b in enumerate(code): phase[i*chip:(i+1)*chip] = 0 if b==1 else np.pi
        return np.exp(1j * phase)
    elif name == 'Costas':
        freqs = np.array([2,4,8,5,10,6,7,3,9,1]) * 2e5 # 간격 조정
        hop = int(n_samples / len(freqs))
        sig = np.zeros(n_samples, dtype=complex)
        for i, f in enumerate(freqs): 
            # 중심 주파수 기준으로 -1MHz ~ +1MHz 사이로 배치
            sig[i*hop:(i+1)*hop] = np.exp(1j * 2 * np.pi * (f - 1e6) * t[i*hop:(i+1)*hop])
        return sig
    elif name == 'Frank':
        M = 4; chip = int(n_samples / (M*M)); phase = np.zeros(n_samples)
        for i in range(M):
            for j in range(M): phase[(i*M+j)*chip:((i*M+j)+1)*chip] = 2*np.pi*i*j/M
        return np.exp(1j * phase)

# 500개씩 만들면서 잡음 섞기 (SNR -5dB ~ 5dB 랜덤)
for idx, name in enumerate(classes):
    for _ in range(n_per_class):
        sig = get_signal(name)
        
        # 랜덤 SNR 적용 (-5dB ~ 5dB 사이)
        snr = np.random.uniform(-5, 5) 
        noise = (np.random.randn(len(sig)) + 1j * np.random.randn(len(sig))) / np.sqrt(2)
        power = np.mean(np.abs(sig)**2)
        noise_p = power / (10**(snr/10))
        noisy_sig = sig + np.sqrt(noise_p) * noise
        
        # STFT 이미지 변환 (64x64 크기로 맞춤)
        f, t_spec, Zxx = signal.stft(noisy_sig, fs, nperseg=64)
        img = np.abs(Zxx) # 크기 정보만 사용
        
        # 이미지 크기 정규화 및 리스트 추가 (CNN 입력용)
        # 중요: CNN은 0~1 사이의 값을 좋아합니다.
        img = img / np.max(img) 
        X_data.append(img)
        y_label.append(idx)

# PyTorch 텐서로 변환
X_data = np.array(X_data)[:, np.newaxis, :, :] # (N, 1, 33, 33) 등의 차원 생성
X_tensor = torch.tensor(X_data, dtype=torch.float32)
y_tensor = torch.tensor(y_label, dtype=torch.long)

# 학습용/테스트용 데이터 나누기 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, shuffle=True)

# 데이터 로더 생성 (배치 사이즈 32: 문제를 32개씩 끊어서 품)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)

print(f"데이터 준비 완료! 학습 데이터: {len(X_train)}개, 테스트 데이터: {len(X_test)}개")

# ==========================================
# 2. CNN 모델 설계 (뇌)
# ==========================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1층: 특징 추출
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1) # 입력채널 1 -> 출력 16
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2) # 이미지 크기 절반으로 줄임
        
        # 2층: 더 깊은 특징 추출
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # 입력 16 -> 출력 32
        
        # 3층: 분류 (Flatten -> Dense)
        # 이미지 크기에 따라 flatten 사이즈가 달라짐. 여기서는 자동 계산된 크기 적용 필요
        # 33x33 -> pool -> 16x16 -> pool -> 8x8
        self.fc = nn.Linear(32 * 8 * 9, 4) # 4개 클래스로 분류 (LFM, Barker, Costas, Frank)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # 평평하게 펴기 (Flatten)
        x = self.fc(x)
        return x

model = SimpleCNN()
criterion = nn.CrossEntropyLoss() # 오차 계산 함수
optimizer = optim.Adam(model.parameters(), lr=0.001) # 최적화 도구

# ==========================================
# 3. 학습 시작 (공부)
# ==========================================
print("\n3. CNN 모델 학습을 시작합니다...")
epochs = 10 # 문제집 10번 반복해서 풀기

train_acc_history = []
test_acc_history = []

for epoch in range(epochs):
    model.train()
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        optimizer.zero_grad()   # 이전 계산값 초기화
        outputs = model(inputs) # 모델이 예측
        loss = criterion(outputs, labels) # 정답과 비교해서 오차 계산
        loss.backward()         # 오차를 줄이는 방향으로 수정 (역전파)
        optimizer.step()        # 모델 업데이트
        
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    train_acc = 100 * correct / total
    train_acc_history.append(train_acc)
    
    # 테스트 (중간 점검)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    test_acc = 100 * correct / total
    test_acc_history.append(test_acc)
    
    print(f"Epoch [{epoch+1}/{epochs}] - 학습 정확도: {train_acc:.2f}%, 테스트 정확도: {test_acc:.2f}%")

# ==========================================
# 4. 결과 시각화 (성적표)
# ==========================================
plt.figure(figsize=(10, 5))
plt.plot(train_acc_history, label='Train Accuracy')
plt.plot(test_acc_history, label='Test Accuracy', linestyle='--')
plt.title('CNN Model Training Result')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()
plt.grid(True)
plt.show()

print("\n축하합니다! 첫 번째 딥러닝 레이더 분류 모델이 완성되었습니다.")

OSError: [WinError 1114] DLL 초기화 루틴을 실행할 수 없습니다. Error loading "c:\Users\박세은\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.